# Tideway Discovery API Examples

Helper methods paired with direct `get`/`post`/`patch` calls for discovery endpoints. Reference: https://docs.bmc.com/xwiki/bin/view/IT-Operations-Management/Discovery/BMC-Helix-Discovery/DAAS/Integrating/Using-the-REST-APIs/Endpoints-in-the-REST-API/.


## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.
- Adjust request bodies to match your appliance and the published API.


In [ ]:
import tideway

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
discovery = tw.discovery()

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Discovery status
Helper vs direct calls for the discovery state.


In [ ]:
status_helper = discovery.get_discovery
status_direct = discovery.get('/discovery')

status_calls = {
    '/discovery (helper)': show_response(status_helper, '/discovery'),
    '/discovery (direct GET)': show_response(status_direct, '/discovery'),
}
status_calls


In [ ]:
discovery_patch_body = {
    # 'state': 'start',  # or 'stop'
    # 'reason': 'Maintenance complete',
}

if discovery_patch_body:
    discovery_patch_helper = discovery.patch_discovery(discovery_patch_body)
    discovery_patch_direct = discovery.patch('/discovery', discovery_patch_body)
    discovery_patch_calls = {
        '/discovery (helper PATCH)': show_response(discovery_patch_helper, '/discovery'),
        '/discovery (direct PATCH)': show_response(discovery_patch_direct, '/discovery'),
    }
else:
    discovery_patch_calls = 'Set discovery_patch_body to start/stop discovery'
discovery_patch_calls


## Provider and cloud metadata
Metadata helpers vs direct calls.


In [ ]:
provider_meta_helper = discovery.get_discovery_api_provider_metadata
provider_meta_direct = discovery.get('/discovery/api_provider_metadata')

cloud_meta_helper = discovery.get_discovery_api_cloud_metadata
cloud_meta_direct = discovery.get('/discovery/cloud_metadata')

metadata_calls = {
    '/discovery/api_provider_metadata (helper)': show_response(provider_meta_helper, '/discovery/api_provider_metadata'),
    '/discovery/api_provider_metadata (direct)': show_response(provider_meta_direct, '/discovery/api_provider_metadata'),
    '/discovery/cloud_metadata (helper)': show_response(cloud_meta_helper, '/discovery/cloud_metadata'),
    '/discovery/cloud_metadata (direct)': show_response(cloud_meta_direct, '/discovery/cloud_metadata'),
}
metadata_calls


## Excludes
List and manage excludes via helpers and direct calls.


In [ ]:
excludes_list_helper = discovery.get_discovery_excludes
excludes_list_direct = discovery.get('/discovery/excludes')

exclude_calls = {
    '/discovery/excludes (helper)': show_response(excludes_list_helper, '/discovery/excludes'),
    '/discovery/excludes (direct)': show_response(excludes_list_direct, '/discovery/excludes'),
}

exclude_body = {
    # 'value': '10.0.0.0/24',
    # 'reason': 'Example exclude',
}
exclude_id = ''  # set to an existing exclude id for patch/delete

if exclude_body:
    exclude_post_helper = discovery.post_discovery_exclude(exclude_body)
    exclude_post_direct = discovery.post('/discovery/excludes', exclude_body)
    exclude_calls['/discovery/excludes (helper POST)'] = show_response(exclude_post_helper, '/discovery/excludes')
    exclude_calls['/discovery/excludes (direct POST)'] = show_response(exclude_post_direct, '/discovery/excludes')

if exclude_id and exclude_body:
    exclude_patch_helper = discovery.patch_discovery_exclude(exclude_id, exclude_body)
    exclude_patch_direct = discovery.patch(f"/discovery/excludes/{exclude_id}", exclude_body)
    exclude_calls[f"/discovery/excludes/{exclude_id} (helper PATCH)"] = show_response(exclude_patch_helper, f"/discovery/excludes/{exclude_id}")
    exclude_calls[f"/discovery/excludes/{exclude_id} (direct PATCH)"] = show_response(exclude_patch_direct, f"/discovery/excludes/{exclude_id}")

if exclude_id:
    exclude_delete_helper = discovery.delete_discovery_exclude(exclude_id)
    exclude_delete_direct = discovery.delete(f"/discovery/excludes/{exclude_id}")
    exclude_calls[f"/discovery/excludes/{exclude_id} (helper DELETE)"] = show_response(exclude_delete_helper, f"/discovery/excludes/{exclude_id}")
    exclude_calls[f"/discovery/excludes/{exclude_id} (direct DELETE)"] = show_response(exclude_delete_direct, f"/discovery/excludes/{exclude_id}")

exclude_calls


## Discovery runs
List runs, create or update run state via helper vs direct calls.


In [ ]:
runs_helper = discovery.get_discovery_runs
runs_direct = discovery.get('/discovery/runs')

run_body = {
    # 'type': 'snapshot',
    # 'label': 'Example run',
    # 'endpoints': ['10.0.0.0/24'],
}
run_patch_body = {
    # 'state': 'cancelled',
}
run_id = ''  # set to an existing run id for patching

run_calls = {
    '/discovery/runs (helper)': show_response(runs_helper, '/discovery/runs'),
    '/discovery/runs (direct)': show_response(runs_direct, '/discovery/runs'),
}

if run_body:
    run_post_helper = discovery.post_discovery_run(run_body)
    run_post_direct = discovery.post('/discovery/runs', run_body)
    run_calls['/discovery/runs (helper POST)'] = show_response(run_post_helper, '/discovery/runs')
    run_calls['/discovery/runs (direct POST)'] = show_response(run_post_direct, '/discovery/runs')

if run_id and run_patch_body:
    run_patch_helper = discovery.patch_discovery_run(run_id, run_patch_body)
    run_patch_direct = discovery.patch(f"/discovery/runs/{run_id}", run_patch_body)
    run_calls[f"/discovery/runs/{run_id} (helper PATCH)"] = show_response(run_patch_helper, f"/discovery/runs/{run_id}")
    run_calls[f"/discovery/runs/{run_id} (direct PATCH)"] = show_response(run_patch_direct, f"/discovery/runs/{run_id}")

run_calls


## Run results and inferred devices
Requires a `run_id`; choose result type or inferred kind per docs.


In [ ]:
run_id_for_results = ''  # e.g. '1234567890abcdef'
result_type = 'Success'
inferred_kind = ''  # e.g. 'Host'

if run_id_for_results:
    run_results_helper = discovery.get_discovery_run_results(run_id_for_results, result_type)
    run_results_direct = discovery.get(f"/discovery/runs/{run_id_for_results}/results/{result_type}")

    inferred_helper = discovery.get_discovery_run_inferred(run_id_for_results, inferred_kind)
    inferred_direct = discovery.get(f"/discovery/runs/{run_id_for_results}/inferred" + (f"/{inferred_kind}" if inferred_kind else ''))

    results_calls = {
        f"/discovery/runs/{run_id_for_results}/results/{result_type} (helper)": show_response(run_results_helper, f"/discovery/runs/{run_id_for_results}/results/{result_type}"),
        f"/discovery/runs/{run_id_for_results}/results/{result_type} (direct)": show_response(run_results_direct, f"/discovery/runs/{run_id_for_results}/results/{result_type}"),
        f"/discovery/runs/{run_id_for_results}/inferred (helper)": show_response(inferred_helper, f"/discovery/runs/{run_id_for_results}/inferred"),
        f"/discovery/runs/{run_id_for_results}/inferred (direct)": show_response(inferred_direct, f"/discovery/runs/{run_id_for_results}/inferred"),
    }
else:
    results_calls = 'Set run_id_for_results to inspect results and inferred devices.'
results_calls


## Scheduled runs
List, create, update, or delete scheduled runs (helper vs direct).


In [ ]:
schedules_helper = discovery.get_discovery_run_schedules
schedules_direct = discovery.get('/discovery/runs/scheduled')

schedule_body = {
    # 'label': 'Weekly scan',
    # 'cron': '0 2 * * 1',
    # 'endpoints': ['10.0.0.0/24'],
}
schedule_patch_body = {
    # 'label': 'Updated label',
}
schedule_id = ''  # set to a scheduled run id for patch/delete

schedule_calls = {
    '/discovery/runs/scheduled (helper)': show_response(schedules_helper, '/discovery/runs/scheduled'),
    '/discovery/runs/scheduled (direct)': show_response(schedules_direct, '/discovery/runs/scheduled'),
}

if schedule_body:
    schedule_post_helper = discovery.post_discovery_run_schedule(schedule_body)
    schedule_post_direct = discovery.post('/discovery/runs/scheduled', schedule_body)
    schedule_calls['/discovery/runs/scheduled (helper POST)'] = show_response(schedule_post_helper, '/discovery/runs/scheduled')
    schedule_calls['/discovery/runs/scheduled (direct POST)'] = show_response(schedule_post_direct, '/discovery/runs/scheduled')

if schedule_id and schedule_patch_body:
    schedule_patch_helper = discovery.patch_discovery_run_schedule(schedule_id, schedule_patch_body)
    schedule_patch_direct = discovery.patch(f"/discovery/runs/scheduled/{schedule_id}", schedule_patch_body)
    schedule_calls[f"/discovery/runs/scheduled/{schedule_id} (helper PATCH)"] = show_response(schedule_patch_helper, f"/discovery/runs/scheduled/{schedule_id}")
    schedule_calls[f"/discovery/runs/scheduled/{schedule_id} (direct PATCH)"] = show_response(schedule_patch_direct, f"/discovery/runs/scheduled/{schedule_id}")

if schedule_id:
    schedule_delete_helper = discovery.delete_discovery_run_schedule(schedule_id)
    schedule_delete_direct = discovery.delete(f"/discovery/runs/scheduled/{schedule_id}")
    schedule_calls[f"/discovery/runs/scheduled/{schedule_id} (helper DELETE)"] = show_response(schedule_delete_helper, f"/discovery/runs/scheduled/{schedule_id}")
    schedule_calls[f"/discovery/runs/scheduled/{schedule_id} (direct DELETE)"] = show_response(schedule_delete_direct, f"/discovery/runs/scheduled/{schedule_id}")

schedule_calls


## Outposts
List, create, or delete Outposts via helpers vs direct calls.


In [ ]:
outposts_helper = discovery.get_discovery_outposts
outposts_direct = discovery.get('/discovery/outposts')

outpost_body = {
    # 'name': 'example-outpost',
    # 'address': 'outpost.example.com',
}
outpost_id = ''  # set to an existing outpost id for delete

outpost_calls = {
    '/discovery/outposts (helper)': show_response(outposts_helper, '/discovery/outposts'),
    '/discovery/outposts (direct)': show_response(outposts_direct, '/discovery/outposts'),
}

if outpost_body:
    outpost_post_helper = discovery.post_discovery_outpost(outpost_body)
    outpost_post_direct = discovery.post('/discovery/outposts', outpost_body)
    outpost_calls['/discovery/outposts (helper POST)'] = show_response(outpost_post_helper, '/discovery/outposts')
    outpost_calls['/discovery/outposts (direct POST)'] = show_response(outpost_post_direct, '/discovery/outposts')

if outpost_id:
    outpost_delete_helper = discovery.delete_discovery_outpost(outpost_id)
    outpost_delete_direct = discovery.delete(f"/discovery/outposts/{outpost_id}")
    outpost_calls[f"/discovery/outposts/{outpost_id} (helper DELETE)"] = show_response(outpost_delete_helper, f"/discovery/outposts/{outpost_id}")
    outpost_calls[f"/discovery/outposts/{outpost_id} (direct DELETE)"] = show_response(outpost_delete_direct, f"/discovery/outposts/{outpost_id}")

outpost_calls
